这个NootBook用于读取手足口病数据，对数据进行必要的预处理，包括转换数据格式、检测和填充空缺数据，检测和替换异常数据。

读取手足口病数据，进行数据类型转化。

In [1]:
#读取手足口病数据
import pandas as pd
HFMD=pd.read_csv('/root/AnalysisData/HFMD.csv')
HFMD.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120138 entries, 0 to 120137
Data columns (total 6 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   Onset Date               120138 non-null  object
 1   Date of Birth            4699 non-null    object
 2   Gender                   120138 non-null  object
 3   Age                      120138 non-null  object
 4   Regional Distribution    120138 non-null  object
 5   Population distribution  120138 non-null  object
dtypes: object(6)
memory usage: 5.5+ MB


In [3]:
#将Onset Date转化为Dateime类型
OnesetData=pd.to_datetime(HFMD.loc[:,'Onset Date'])
HFMD['Oneset_Data']=OnesetData

In [4]:
#将中文性别转化为英文，并对其进行编码：男：1，女：0；
ChintoEln_gender={'男':'Male','女':'Female'}
GenderEncoder={'Male':1,'Female':0}
HFMD['GenderToEln']=HFMD['Gender'].map(ChintoEln_gender)
HFMD['GenderEncoder']=HFMD['GenderToEln'].map(GenderEncoder)

In [5]:
HFMD.head(10)

,Onset Date,Date of Birth,Gender,Age,Regional Distribution,Population distribution,Oneset_Data,GenderToEln,GenderEncoder
0,2018/1/16,2015/4/1,女,3岁,花溪区,散居儿童,2018-01-16,Female,0
1,2018/4/2,2016/10/28,女,2岁,花溪区,散居儿童,2018-04-02,Female,0
2,2018/4/2,2012/4/6,男,1岁,观山湖区,散居儿童,2018-04-02,Male,1
3,2018/4/11,2014/3/13,女,1岁,本省其他市州,散居儿童,2018-04-11,Female,0
4,2018/4/11,2013/8/21,女,4岁,花溪区,幼托儿童,2018-04-11,Female,0
5,2018/4/11,2015/1/1,男,2岁,观山湖区,散居儿童,2018-04-11,Male,1
6,2018/4/13,2016/6/10,男,4岁,乌当区,散居儿童,2018-04-13,Male,1
7,2018/4/14,2016/5/28,女,8月,南明区,散居儿童,2018-04-14,Female,0
8,2018/4/14,2015/7/2,女,1岁,花溪区,散居儿童,2018-04-14,Female,0
9,2018/4/21,2014/8/31,男,2岁,云岩区,散居儿童,2018-04-21,Male,1


In [28]:
#计算年龄
from pandas import DataFrame
import re
#提取字符串中的纯数字并转换为整数
def extract_age_numeric(age_str):
    cleaned_str = age_str.replace(",", "").replace(" ", "")
    age_matches = re.findall(r'(\d+)(岁|月|天)', cleaned_str)
    age_dict = {"岁": 0, "月": 0, "天": 0}
    for numeric, unit in age_matches:
        age_dict[unit] = int(numeric)
    return age_dict
ComputerAge=DataFrame()
age_structured=HFMD["Age"].apply(extract_age_numeric)
ComputerAge["year"] = [item["岁"] for item in age_structured]
ComputerAge["month"] = [item["月"] for item in age_structured]
ComputerAge["Day"] = [item["天"] for item in age_structured]

In [29]:
# 将岁、月、天统一换算为「岁」
def convert_to_years_total(row):
    years = row["year"]
    months = row["month"]
    days = row["Day"]
    #月→岁，天→岁，累加得到总年龄
    years_from_months = months / 12  # 月份换算为岁
    years_from_days = days / 365.25  # 天数直接换算为岁
    total_years = years + years_from_months + years_from_days
    return round(total_years, 2)
ComputerAge["ComputerAge"]=ComputerAge.apply(convert_to_years_total, axis=1)

In [30]:
#年龄整合
HFMD['ComputerAge']=ComputerAge["ComputerAge"]

In [16]:
#将地区转化为英文
ChinToEln_Dir={'云岩区':'Yunyan District','南明区':'Nanming District','花溪区':'Huaxi District',
           '观山湖区':'Guanshanhu District', '乌当区':'Wudang District','本省其他市州':'Other cities and states in the province ',
           '修文县':'Xiuwen County','白云区':'Baiyun District','开阳县':'Kaiyang County','息烽县':'Xifeng County',
           '清镇市':'Qingzhen City','外省':'Outside the province'}
HFMD['Regional']=HFMD.loc[:,'Regional Distribution'].map(ChinToEln_Dir)

In [20]:
# 中文-英文对应字典（严格遵循要求：所有"其他/其它"英文统一为"Other"）
occupation_dict = {"散居儿童": "Migrant Children","幼托儿童": "Nursery and Kindergarten Children","学生": "Student","不详": "Unknown",
                       "家务及待业": "Housework and Unemployed","干部职员": "Cadres and Staff","农民": "Farmer","商业服务": "Commercial Services",
                       "工人": "Worker","教师": "Teacher","医务人员": "Medical Personnel","其他:散居儿童": "Other: Migrant Children","其它:公司职员": "Other: Company Staff",
                       "离退人员": "Retired Personnel","民工": "Migrant Worker","其它:医院工作人员": "Other","其它:散居儿童": "Other","其它:个体": "Other",
                       "其他:公司职员": "Other","其他:幼托儿童": "Other","保育员及保姆": "Nursery Nurse and Maid","其它:幼托儿童": "Other","餐饮食品业": "Catering and Food Industry","其它:公务员": "Other","其它:实验室工作": "Other","其它:工程技术人员": "Other",
                       "其它:驾驶员": "Other","牧民": "Herdsman","其它:企业管理人员": "Other","其它:无": "Other","其它:销售": "Other","其它:电网公司职员": "Other","其它:职业学校工作人员": "Other",
                       "其它:银行职员": "Other","其它:收费处工作人员": "Other","其它:保洁": "Other","其它:行政办公人员": "Other","其它:个体销售": "Other","其它:学校工作人员": "Other","其它:马铃乡文化服务中心工作人员": "Other",
                       "其它:运输服务人员": "Other","其它:个体户": "Other","其它:工人": "Other","公共场所服务员": "Public Place Attendant","其他:父亲身份证号": "Other","其他:职员": "Other",
                       "渔(船)民": "Fisherman","其他:妈妈身份证号": "Other"}
HFMD['Population']=HFMD.loc[:,'Population distribution'].map(occupation_dict)

计算每个月发病率(发病人口数/10万)
计算方法：发病率=发病人口数（月）/年平均人口数
总人口数:4751940(2018)、4817582（2019）、4869623（2020）、5851742（2021）、5931912（2022）

In [42]:
#以月份为单位统计每个月疾病报告次数
monthly_count=HFMD["Oneset_Data"].dt.to_period('M').value_counts().sort_index()
complete_date_range=pd.date_range(start="2018-01-01", end="2022-12-31", freq="MS")  # MS：每月第一天
complete_month_index=complete_date_range.to_period('M')
monthly_count_complete=monthly_count.reindex(complete_month_index, fill_value=0)
monthly_count_complete_df=monthly_count_complete.reset_index()
monthly_count_complete_df.columns =["year_month", "counts"]

In [57]:
#统计每年患病人口数
sum_2018=monthly_count_complete_df.iloc[:12,1].sum()
sum_2019=monthly_count_complete_df.iloc[12:24,1].sum()
sum_2020=monthly_count_complete_df.iloc[24:36,1].sum()
sum_2021=monthly_count_complete_df.iloc[36:48,1].sum()
sum_2022=monthly_count_complete_df.iloc[48:,1].sum()
print(f"年患者数:2018年{sum_2018},2019年{sum_2019},2020年{sum_2020},2021年{sum_2021},2022年{sum_2022}")

年患者数:2018年36457,2019年36490,2020年16932,2021年15183,2022年15076


In [58]:
#计算2018年-2022年每个月HFMD发病率
IR_2018=(monthly_count_complete_df.iloc[:12,1]/4751940)*100000    #2018年
IR_2019=(monthly_count_complete_df.iloc[12:24,1]/4817582)*100000    #2019年
IR_2020=(monthly_count_complete_df.iloc[24:36,1]/4869623)*100000    #2020年
IR_2021=(monthly_count_complete_df.iloc[36:48,1]/5851742)*100000    #2021年
IR_2022=(monthly_count_complete_df.iloc[48:,1]/5931912)*100000    #2022年

In [59]:
#验证发病率
print(f"年发病率:2018年{IR_2018.sum()},2019年{IR_2019.sum()},2020年{IR_2020.sum()},2021年{IR_2021.sum()},2022年{IR_2022.sum()}")

年发病率:2018年767.2024478423549,2019年757.4339160184508,2020年347.7065883745004,2021年259.4611997589778,2022年254.15076960008852


In [72]:
#合并年发病率并添加至气象数据中
IR_months=pd.concat([IR_2018,IR_2019,IR_2020,IR_2021,IR_2022],axis=0)
ClimData=pd.read_csv('/root/AnalysisData/Meteorological Data.csv')
ClimData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date               60 non-null     object 
 1   Pressure(hPa)      60 non-null     float64
 2   RH(%)              60 non-null     float64
 3   WindSpeed(m/s)     60 non-null     float64
 4   temperature_mean   60 non-null     float64
 5   temperature_min    60 non-null     float64
 6   temperature_max    60 non-null     float64
 7   petPM(mm)          60 non-null     float64
 8   precipitation(mm)  60 non-null     float64
dtypes: float64(8), object(1)
memory usage: 4.3+ KB


In [74]:
#合并发病率数据
ClimData['IR']=IR_months

In [79]:
#将气象数据与发病率数据合并
ClimData.to_csv('/root/AnalysisData/AnalysisData.csv',encoding='utf-8')

In [89]:
#输出整理后的HFMD数据
HFMDProcess=HFMD.iloc[:,6:-1]
HFMDProcess.to_csv('/root/AnalysisData/HFMDProcess.csv',encoding='utf-8')